# CogMem Phase 1 — Qwen2.5:3b Episode Collection\n\nCollect Q-valued episodes from MemRL on ALFWorld using **Qwen2.5:3b**.\nThe model's own meta-reflections will be used for Phase 2 training (no expert data needed).\n\n**Flow:** Cells 1-9 sequentially. If notebook restarts, run 1-4, then 9b, then 9.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# Cell 2: Install system deps + Ollama
!apt-get update -qq && apt-get install -y -qq zstd cmake build-essential > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Cell 3: Start Ollama + pull models
import subprocess, time
proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**__import__("os").environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text
print("Ollama ready!")

In [ ]:
# Cell 4: Install ALFWorld + MemRL
!pip install alfworld -q && alfworld-download
!cd /notebooks && git clone https://github.com/MemTensor/MemRL 2>/dev/null
!cd /notebooks/MemRL && pip install -e . -q
!pip install "chonkie==1.2.1" qdrant-client tensorboard tiktoken openai textworld -q
!pip install "pydantic>=2.0" "transformers>=4.35,<4.38" -q
!python3 -c "import memos; import alfworld; print('All imports OK')"

In [ ]:
# Cell 5: Patch MemRL — fix vector dimension + save trajectories
import json

# Fix embedding dimension (768 for nomic-embed-text)
path = "/notebooks/MemRL/memrl/service/memory_service.py"
with open(path) as f:
    code = f.read()
code = code.replace('"vector_dimension": 3072', '"vector_dimension": 768')
if "trajectories.jsonl" not in code:
    old = '                logger.info(f"Mini-batch {i+1} collected {len(collected_trajs)} trajectories.")\n                section_trajectories.extend(collected_trajs)\n\n                # --- Memory Processing for this mini-batch ---'
    new = '''                logger.info(f"Mini-batch {i+1} collected {len(collected_trajs)} trajectories.")
                section_trajectories.extend(collected_trajs)

                # --- Save raw trajectories to disk ---
                traj_save_path = self.local_cache_dir / "trajectories.jsonl"
                try:
                    with open(traj_save_path, "a", encoding="utf-8") as f:
                        for traj in collected_trajs:
                            record = {
                                "task_description": traj["task_description"],
                                "success": traj["success"],
                                "steps": int(traj["steps"]) if hasattr(traj["steps"], "item") else traj["steps"],
                                "gamefile": traj.get("gamefile", ""),
                                "trajectory": self._sanitize_reflection_trajectory(traj["trajectory"]),
                            }
                            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\\n")
                except Exception:
                    logger.warning("Failed to save trajectories", exc_info=True)

                # --- Memory Processing for this mini-batch ---'''
    code = code.replace(old, new)
with open(path, "w") as f:
    f.write(code)
print("MemRL patched: vector_dimension=768, trajectory saving enabled")

In [ ]:
# Cell 6: Setup ALFWorld data
import os, shutil
from pathlib import Path

memrl = Path("/notebooks/MemRL")
alf_data = memrl / "data" / "alfworld"
alf_data.mkdir(parents=True, exist_ok=True)
cache = Path("/root/.cache/alfworld")
for name in ["json_2.1.1", "logic", "detectors"]:
    src, dst = cache / name, alf_data / name
    if src.exists() and not dst.exists():
        os.symlink(str(src), str(dst))
src_ex = memrl / "configs" / "alfworld" / "alfworld_examples.json"
dst_ex = alf_data / "alfworld_examples.json"
if src_ex.exists() and not dst_ex.exists():
    shutil.copy2(src_ex, dst_ex)
print("ALFWorld data ready!")

In [ ]:
# Cell 7: Config — Qwen2.5:3b, 248 games, 1 section
config = """
llm:
  provider: "openai"
  api_key: "ollama"
  base_url: "http://localhost:11434/v1"
  model: "qwen2.5:3b"
  temperature: 0
  max_tokens: 4096

embedding:
  provider: "openai"
  api_key: "ollama"
  base_url: "http://localhost:11434/v1"
  model: "nomic-embed-text"
  max_text_len: 4096

memory:
  build_strategy: "proceduralization"
  retrieve_strategy: "query"
  update_strategy: "adjustment"
  k_retrieve: 5
  max_keywords: 8
  confidence_threshold: 0.0
  memory_confidence: 100.0
  add_similarity_threshold: 0.90
  mos_config_path: "configs/mos_config_final.json"
  user_id: "memrl_user"
  sim_norm_mean: 0.5187
  sim_norm_std: 0.1203

environment:
  alfworld_config_path: "configs/envs/alfworld.yaml"
  alfworld_env_type: "AlfredTWEnv"

experiment:
  random_seed: 42
  enable_value_driven: true
  experiment_name: 'cogmem_phase1_qwen'
  mode: 'train'
  num_sections: 1
  batch_size: 8
  dataset_ratio: 0.07
  few_shot_path: 'data/alfworld/alfworld_examples.json'
  max_steps: 20
  bon: 1
  valid_interval: 1
  test_interval: 5
  baseline_mode: none
  baseline_k: 10
  output_dir: "./results"
  save_trajectories: true
  save_memories: true
  ckpt_eval_enabled: false
  ckpt_eval_path: ""
  ckpt_resume_enabled: true
  ckpt_resume_path: ""
  ckpt_resume_epoch: null

rl_config:
  epsilon: 0
  tau: 0.62
  alpha: 0.3
  gamma: 0.0
  q_init_pos: 0
  q_init_neg: 0
  success_reward: 1.0
  failure_reward: -1.0
  topk: 3
  novelty_threshold: 0.85
  recency_boost: 0.0
  reward_merge_gain: 0.1
  q_min_threshold: -10
  weight_sim: 0.5
  weight_q: 0.5
""".strip()

with open("/notebooks/MemRL/configs/rl_alf_config.local.yaml", "w") as f:
    f.write(config)
print("Config: qwen2.5:3b, ~248 games, 1 section, checkpoint resume ON")

In [ ]:
# Cell 8: Quick test
!curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:3b","prompt":"Say OK","stream":false}' | python3 -c "import sys,json; d=json.load(sys.stdin); print('Ollama OK:', d['response'][:30])"
!python3 -c "import alfworld; print('ALFWorld OK')"
!cd /notebooks/MemRL && python3 -c "from memrl.configs.config import MempConfig; cfg = MempConfig.from_yaml('configs/rl_alf_config.local.yaml'); print(f'MemRL OK: {cfg.llm.model}, batch={cfg.experiment.batch_size}')"

In [ ]:
# Cell 9: RUN Phase 1
!cd /notebooks/MemRL && python3 run/run_alfworld.py --config configs/rl_alf_config.local.yaml

In [ ]:
# Cell 9b: RESUME after restart — run cells 2-4 first, then this, then cell 9
import glob, os, yaml

ckpt_dirs = sorted(glob.glob("/notebooks/MemRL/results/alfworld/exp_cogmem_phase1_qwen_*/local_cache"), key=os.path.getmtime)
if ckpt_dirs:
    latest = ckpt_dirs[-1]
    print(f"Found checkpoint: {latest}")
    config_path = "/notebooks/MemRL/configs/rl_alf_config.local.yaml"
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    cfg["experiment"]["ckpt_resume_enabled"] = True
    cfg["experiment"]["ckpt_resume_path"] = latest
    with open(config_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("Config updated! Now re-run Cell 9")
else:
    print("No checkpoint found — run Cell 9 from scratch")

In [ ]:
# Cell 10: Check results + package for download
!echo "=== Cube dumps ==="
!find /notebooks/MemRL/results -name "textual_memory.json" -size +10c -exec ls -lh {} \;
!echo "=== Trajectories ==="
!find /notebooks/MemRL/results -name "trajectories.jsonl" -exec wc -l {} \;
!echo "=== Games finished ==="
!grep -c "finished a game" /notebooks/MemRL/logs/cogmem_phase1_qwen/*.log 2>/dev/null
!echo "=== Successes ==="
!grep -c "Success: True" /notebooks/MemRL/logs/cogmem_phase1_qwen/*.log 2>/dev/null

# Package results
!cd /notebooks/MemRL && tar czf /notebooks/cogmem_phase1_qwen.tar.gz \
    $(find results -name "textual_memory.json" -size +10c 2>/dev/null) \
    $(find results -name "trajectories.jsonl" 2>/dev/null) \
    $(find results -name "snapshot_meta.json" 2>/dev/null) \
    2>/dev/null
!ls -lh /notebooks/cogmem_phase1_qwen.tar.gz 2>/dev/null
print("Download cogmem_phase1_qwen.tar.gz")